# 04 — Spécificités Oracle : PL/SQL, DUAL, ROWNUM, Séquences


## Sommaire
1. Setup — générer le script de création de la base Oracle
2. Connecter Oracle XE à DataGrip
3. `DUAL` — la pseudo-table Oracle
4. `ROWNUM` vs `FETCH FIRST` — un piège classique
5. Séquences — l'auto-incrément à l'oracle
6. PL/SQL — blocs anonymes
7. Curseurs explicites
8. Fonctions stockées
9. Procédures stockées (paramètres IN/OUT)
10. Gestion des exceptions
11. Window functions — ce qui ne change pas
12. Récapitulatif comparatif SQLite/standard SQL ↔ Oracle
13. Bonus — connexion Python à Oracle (python-oracledb)


## 1. Setup — générer le script de création de la base Oracle

On reprend exactement le même jeu de données que dans `02_sql_advanced.ipynb` et `03_pandas_advanced.ipynb` (communes normandes + relevés de pollution) — traduit en syntaxe Oracle. Cette cellule génère le script complet en un fichier `.sql`, à exécuter tel quel dans la console DataGrip connectée à ta base Oracle.

In [1]:
# Génère le script SQL complet (DDL + DML) en syntaxe Oracle
# Mêmes données exactes que 02_sql_advanced.ipynb / 03_pandas_advanced.ipynb

communes_data = [
    (1, "Le Havre",   "Seine-Maritime", 170000, 0.9041, 21.4),
    (2, "Rouen",      "Seine-Maritime", 110000, 0.5451, 15.2),
    (3, "Dieppe",     "Seine-Maritime",  30000, 0.5500, 16.7),
    (4, "Caen",       "Calvados",       105000, 0.1771,  9.8),
    (5, "Evreux",     "Eure",            48000, 0.4200, 13.1),
    (6, "Cherbourg",  "Manche",          78000, 0.3100, 11.0),
]

import numpy as np
np.random.seed(42)

base_no2 = {1: 38.2, 2: 27.5, 3: 25.9, 4: 14.1, 5: 22.0, 6: 18.4}
mois_liste = [f"2025-{m:02d}" for m in range(1, 13)]

releves_rows = []
for commune_id, base in base_no2.items():
    for i, mois in enumerate(mois_liste):
        saison = 6 * np.cos(2 * np.pi * i / 12)
        bruit = np.random.normal(0, 1.5)
        valeur = round(base + saison + bruit, 1)
        releves_rows.append((commune_id, mois, valeur))

script_lines = []
script_lines.append("-- =========================================================")
script_lines.append("-- EcoSense / SQL Practice -- Script de setup Oracle (04)")
script_lines.append("-- A executer dans DataGrip, connecte a ta base Oracle XE")
script_lines.append("-- =========================================================")
script_lines.append("")
script_lines.append("DROP TABLE releves_pollution;")
script_lines.append("DROP TABLE communes;")
script_lines.append("DROP SEQUENCE releve_seq;")
script_lines.append("")
script_lines.append("CREATE TABLE communes (")
script_lines.append("    commune_id     NUMBER PRIMARY KEY,")
script_lines.append("    nom            VARCHAR2(50) NOT NULL,")
script_lines.append("    region         VARCHAR2(50) NOT NULL,")
script_lines.append("    population     NUMBER,")
script_lines.append("    score_je       NUMBER(6,4),")
script_lines.append("    taux_pauvrete  NUMBER(5,2)")
script_lines.append(");")
script_lines.append("")

for row in communes_data:
    script_lines.append(
        f"INSERT INTO communes VALUES ({row[0]}, '{row[1]}', '{row[2]}', {row[3]}, {row[4]}, {row[5]});"
    )

script_lines.append("")
script_lines.append("CREATE SEQUENCE releve_seq START WITH 1 INCREMENT BY 1;")
script_lines.append("")
script_lines.append("CREATE TABLE releves_pollution (")
script_lines.append("    releve_id     NUMBER PRIMARY KEY,")
script_lines.append("    commune_id    NUMBER,")
script_lines.append("    mois          VARCHAR2(7),")
script_lines.append("    no2           NUMBER(5,1),")
script_lines.append("    CONSTRAINT fk_commune FOREIGN KEY (commune_id) REFERENCES communes(commune_id)")
script_lines.append(");")
script_lines.append("")

for commune_id, mois, no2 in releves_rows:
    script_lines.append(
        f"INSERT INTO releves_pollution VALUES (releve_seq.NEXTVAL, {commune_id}, '{mois}', {no2});"
    )

script_lines.append("")
script_lines.append("COMMIT;")

script = "\n".join(script_lines)

with open("04_oracle_setup.sql", "w", encoding="utf-8") as f:
    f.write(script)

print(f"Script genere : 04_oracle_setup.sql ({len(script_lines)} lignes, {len(communes_data)} communes, {len(releves_rows)} releves)")
print()
print("--- Apercu des 15 premieres lignes ---")
print("\n".join(script_lines[:15]))


Script genere : 04_oracle_setup.sql (109 lignes, 6 communes, 72 releves)

--- Apercu des 15 premieres lignes ---
-- =========================================================
-- EcoSense / SQL Practice -- Script de setup Oracle (04)
-- A executer dans DataGrip, connecte a ta base Oracle XE
-- =========================================================

DROP TABLE releves_pollution;
DROP TABLE communes;
DROP SEQUENCE releve_seq;

CREATE TABLE communes (
    commune_id     NUMBER PRIMARY KEY,
    nom            VARCHAR2(50) NOT NULL,
    region         VARCHAR2(50) NOT NULL,
    population     NUMBER,
    score_je       NUMBER(6,4),


## 2. Connecter Oracle XE à DataGrip

Deux options pour avoir une base Oracle disponible, de la plus simple à la plus lourde :

**Option A — Docker (recommandé, rapide)**
```bash
docker run -d --name oracle-xe -p 1521:1521 -e ORACLE_PASSWORD=ton_mdp gvenzl/oracle-xe:21-slim
```
Attends 1-2 minutes que le conteneur finisse son initialisation (`docker logs -f oracle-xe`, cherche `DATABASE IS READY TO USE`).

**Option B — Installation native**
Télécharger Oracle Database XE directement depuis [oracle.com](https://www.oracle.com/database/technologies/xe-downloads.html) — plus lourd, plus long à configurer, mais pas de dépendance à Docker.

**Connexion depuis DataGrip**
1. `File > New > Data Source > Oracle`
2. Host : `localhost`, Port : `1521`
3. Service name : `XEPDB1` (pluggable database par défaut de l'image Docker ci-dessus)
4. User / Password : ceux définis à la création (`system` / `ton_mdp` avec l'option Docker)
5. Teste la connexion, puis ouvre une nouvelle console SQL sur cette connexion pour y exécuter `04_oracle_setup.sql` généré ci-dessus.


## 3. `DUAL` — la pseudo-table Oracle

Contrairement à SQLite ou PostgreSQL, **Oracle exige toujours une clause `FROM`**, même pour une simple expression sans table réelle. `DUAL` est une table système à une seule ligne et une seule colonne, prévue exactement pour cet usage.

```sql
-- Fonctionne en SQLite/PostgreSQL, PAS en Oracle :
-- SELECT 1 + 1;

-- Syntaxe Oracle obligatoire :
SELECT 1 + 1 FROM DUAL;

SELECT SYSDATE FROM DUAL;              -- date/heure serveur actuelle
SELECT USER FROM DUAL;                 -- utilisateur connecté
SELECT UPPER('le havre') FROM DUAL;    -- tester une fonction isolée
```

> On utilise `DUAL` très souvent pour tester rapidement une expression ou une fonction sans avoir besoin d'une vraie table.


## 4. `ROWNUM` vs `FETCH FIRST` — un piège classique d'entretien

### Le piège

`ROWNUM` est attribué **avant** l'exécution du `ORDER BY` — un piège très classique en entretien Oracle.

```sql
-- PIÈGE : ceci NE retourne PAS les 3 communes au score JE le plus élevé
SELECT nom, score_je
FROM communes
WHERE ROWNUM <= 3
ORDER BY score_je DESC;
-- ROWNUM filtre sur les 3 premières lignes retournées par la requête AVANT le tri,
-- puis SEULEMENT ENSUITE trie ces 3 lignes déjà sélectionnées au hasard de l'ordre physique.
```

### La bonne pratique (Oracle < 12c)

```sql
-- Correct : sous-requête pour trier D'ABORD, puis limiter
SELECT * FROM (
    SELECT nom, score_je
    FROM communes
    ORDER BY score_je DESC
)
WHERE ROWNUM <= 3;
```

### La syntaxe moderne (Oracle 12c+, recommandée)

```sql
-- Plus lisible, pas de piège possible
SELECT nom, score_je
FROM communes
ORDER BY score_je DESC
FETCH FIRST 3 ROWS ONLY;

-- Avec pagination (équivalent OFFSET) :
SELECT nom, score_je
FROM communes
ORDER BY score_je DESC
OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY;
```

> Équivalent SQLite : `... ORDER BY score_je DESC LIMIT 3` — bien plus simple, mais Oracle ne l'a introduit (sous la forme `FETCH FIRST`) qu'à partir de la version 12c.


## 5. Séquences — l'auto-incrément à l'oracle

SQLite gère l'auto-incrément avec `INTEGER PRIMARY KEY AUTOINCREMENT` directement sur la colonne. Oracle (avant la version 12c) sépare ça en un objet indépendant, la **séquence**, qu'on appelle explicitement à chaque insertion.

```sql
CREATE SEQUENCE releve_seq START WITH 1 INCREMENT BY 1;

INSERT INTO releves_pollution (releve_id, commune_id, mois, no2)
VALUES (releve_seq.NEXTVAL, 1, '2025-01', 41.3);

-- Consulter la dernière valeur générée dans la session en cours
SELECT releve_seq.CURRVAL FROM DUAL;
```

**Alternative moderne (Oracle 12c+)** — colonne `IDENTITY`, plus proche de la syntaxe SQLite/PostgreSQL :

```sql
CREATE TABLE releves_pollution (
    releve_id  NUMBER GENERATED ALWAYS AS IDENTITY,
    commune_id NUMBER,
    mois       VARCHAR2(7),
    no2        NUMBER(5,1)
);
-- Plus besoin d'appeler .NEXTVAL manuellement à l'insertion
```

> Le script généré en section 1 utilise volontairement la séquence "à l'ancienne" (`releve_seq.NEXTVAL`) — c'est encore ce que tu croiseras le plus souvent en entreprise sur des bases Oracle historiques.


## 6. PL/SQL — blocs anonymes

PL/SQL est le langage procédural propriétaire d'Oracle, qui encapsule du SQL dans une structure `DECLARE / BEGIN / END`. C'est la plus grosse différence avec SQLite (qui ne fait "que" du SQL déclaratif, sans variables ni logique procédurale).

```sql
SET SERVEROUTPUT ON;   -- active l'affichage de DBMS_OUTPUT dans la console

DECLARE
    v_nom    communes.nom%TYPE;       -- %TYPE : hérite automatiquement du type de la colonne
    v_score  communes.score_je%TYPE;
BEGIN
    SELECT nom, score_je
    INTO v_nom, v_score               -- INTO : obligatoire pour stocker un résultat SELECT
    FROM communes
    WHERE commune_id = 1;

    DBMS_OUTPUT.PUT_LINE('Commune : ' || v_nom || ' - Score JE : ' || v_score);
END;
/
```

**Points clés à retenir** :
- `%TYPE` évite de dupliquer/désynchroniser un type de donnée entre la table et la variable
- `INTO` est obligatoire pour affecter le résultat d'un `SELECT` à des variables
- `DBMS_OUTPUT.PUT_LINE` est l'équivalent d'un `print()` — nécessite `SET SERVEROUTPUT ON` pour être visible dans la console
- Le `/` final déclenche l'exécution du bloc dans SQL*Plus / DataGrip


## 7. Curseurs explicites

Un curseur permet de parcourir un ensemble de résultats **ligne par ligne** en PL/SQL — utile quand une opération doit être appliquée individuellement à chaque ligne (chose qu'un simple SQL déclaratif ne permet pas).

```sql
SET SERVEROUTPUT ON;

DECLARE
    CURSOR c_communes IS
        SELECT nom, score_je
        FROM communes
        ORDER BY score_je DESC;

    v_nom    communes.nom%TYPE;
    v_score  communes.score_je%TYPE;
BEGIN
    OPEN c_communes;
    LOOP
        FETCH c_communes INTO v_nom, v_score;
        EXIT WHEN c_communes%NOTFOUND;    -- condition de sortie de boucle

        DBMS_OUTPUT.PUT_LINE(v_nom || ' : ' || v_score);
    END LOOP;
    CLOSE c_communes;   -- toujours refermer le curseur explicitement
END;
/
```

> En pratique, si l'objectif est juste d'afficher ou d'agréger des données, une requête SQL déclarative classique (comme dans `02_sql_advanced.ipynb`) est presque toujours préférable — plus rapide et plus lisible. Les curseurs deviennent utiles quand chaque ligne déclenche une action complexe (appel d'une procédure, écriture conditionnelle ailleurs, etc.), pas pour une simple lecture.


## 8. Fonctions stockées

Une fonction stockée encapsule une logique réutilisable directement en base — utilisable ensuite comme n'importe quelle fonction SQL native, dans un `SELECT`.

```sql
CREATE OR REPLACE FUNCTION categoriser_severite (p_score IN NUMBER)
RETURN VARCHAR2
IS
BEGIN
    IF p_score >= 0.7 THEN
        RETURN 'CRITIQUE';
    ELSIF p_score >= 0.35 THEN
        RETURN 'MODÉRÉ';
    ELSE
        RETURN 'BON';
    END IF;
END;
/

-- Utilisation directe dans une requête
SELECT nom, score_je, categoriser_severite(score_je) AS severite
FROM communes
ORDER BY score_je DESC;
```

> **Fil rouge avec `03_pandas_advanced.ipynb`** : cette fonction reproduit exactement la logique de `categoriser_severite()` écrite en Python avec `apply()` dans le notebook 03, et le `CASE WHEN` du notebook SQL classique. Trois façons différentes d'exprimer la même règle métier, selon la couche (base, procédural, Python) où elle vit.


## 9. Procédures stockées (paramètres IN/OUT)

Contrairement à une fonction (qui retourne toujours une seule valeur via `RETURN`), une procédure peut renvoyer **plusieurs résultats** via des paramètres `OUT`, ou n'en renvoyer aucun (simple exécution d'actions).

```sql
CREATE OR REPLACE PROCEDURE stats_region (
    p_region        IN  communes.region%TYPE,
    p_moyenne       OUT NUMBER,
    p_nb_communes   OUT NUMBER
)
IS
BEGIN
    SELECT AVG(score_je), COUNT(*)
    INTO p_moyenne, p_nb_communes
    FROM communes
    WHERE region = p_region;
END;
/

-- Appel depuis un bloc anonyme
SET SERVEROUTPUT ON;
DECLARE
    v_moy NUMBER;
    v_nb  NUMBER;
BEGIN
    stats_region('Seine-Maritime', v_moy, v_nb);
    DBMS_OUTPUT.PUT_LINE('Moyenne : ' || ROUND(v_moy, 3) || ' sur ' || v_nb || ' communes');
END;
/
```


## 10. Gestion des exceptions

PL/SQL propose un vrai mécanisme de gestion d'erreurs structuré, avec des exceptions nommées prédéfinies par Oracle (`NO_DATA_FOUND`, `TOO_MANY_ROWS`, `ZERO_DIVIDE`...) en plus des exceptions personnalisées.

```sql
SET SERVEROUTPUT ON;

DECLARE
    v_score communes.score_je%TYPE;
BEGIN
    SELECT score_je INTO v_score
    FROM communes
    WHERE commune_id = 999;   -- cet ID n'existe pas

EXCEPTION
    WHEN NO_DATA_FOUND THEN
        DBMS_OUTPUT.PUT_LINE('Aucune commune trouvée avec cet ID.');
    WHEN TOO_MANY_ROWS THEN
        DBMS_OUTPUT.PUT_LINE('Plusieurs lignes trouvées, SELECT INTO en attend une seule.');
    WHEN OTHERS THEN
        DBMS_OUTPUT.PUT_LINE('Erreur inattendue : ' || SQLERRM);
END;
/
```

> `SELECT ... INTO` lève automatiquement `NO_DATA_FOUND` si aucune ligne ne correspond, et `TOO_MANY_ROWS` si plusieurs lignes sont trouvées (une variable scalaire ne peut recevoir qu'UNE seule valeur) — un comportement à connaître, il surprend souvent en entretien.


## 11. Window functions — ce qui ne change pas

Oracle a été l'un des premiers SGBD à implémenter les window functions, et leur syntaxe est quasi identique à celle vue dans `02_sql_advanced.ipynb` (`ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `PARTITION BY`, `LAG()`/`LEAD()`, `ROWS BETWEEN ... PRECEDING AND CURRENT ROW`). Tu peux réutiliser directement ces requêtes contre ta base Oracle sans les réécrire.

**Seule différence à surveiller** : si tu combines une window function avec une limitation de résultats, utilise `FETCH FIRST` (section 4) plutôt que `LIMIT` (qui n'existe pas en Oracle).

```sql
-- Fonctionne à l'identique sur Oracle et SQLite
SELECT
    nom,
    score_je,
    ROW_NUMBER() OVER (ORDER BY score_je DESC) AS rang
FROM communes;
```


## 12. Récapitulatif comparatif — SQLite/standard SQL ↔ Oracle

In [2]:
import pandas as pd

recap = pd.DataFrame([
    ["Sélection sans table",        "SELECT 1+1;",                          "SELECT 1+1 FROM DUAL;"],
    ["Limiter les résultats",       "LIMIT 3",                              "FETCH FIRST 3 ROWS ONLY  (ou ROWNUM avec sous-requête)"],
    ["Pagination",                  "LIMIT 3 OFFSET 3",                     "OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY"],
    ["Type texte",                  "TEXT",                                 "VARCHAR2(n)"],
    ["Type numérique",              "INTEGER / REAL",                       "NUMBER  ou  NUMBER(p,s)"],
    ["Auto-incrément",              "INTEGER PRIMARY KEY AUTOINCREMENT",    "SEQUENCE + .NEXTVAL  (ou IDENTITY en 12c+)"],
    ["Date/heure serveur",          "datetime('now')",                      "SYSDATE  (via DUAL)"],
    ["Bloc procédural",             "Non supporté nativement",              "PL/SQL : DECLARE / BEGIN / END"],
    ["Fonction personnalisée",      "Non supporté nativement",              "CREATE OR REPLACE FUNCTION ... RETURN"],
    ["Procédure stockée",           "Non supporté nativement",              "CREATE OR REPLACE PROCEDURE"],
    ["Window functions",            "Supportées (syntaxe standard)",        "Supportées (syntaxe quasi identique)"],
    ["CTE / WITH",                  "Supportées",                           "Supportées (syntaxe identique)"],
], columns=["Besoin", "SQLite / SQL standard", "Oracle"])

recap


,Besoin,SQLite / SQL standard,Oracle
0,Sélection sans table,SELECT 1+1;,SELECT 1+1 FROM DUAL;
1,Limiter les résultats,LIMIT 3,FETCH FIRST 3 ROWS ONLY (ou ROWNUM avec sous-...
2,Pagination,LIMIT 3 OFFSET 3,OFFSET 3 ROWS FETCH NEXT 3 ROWS ONLY
3,Type texte,TEXT,VARCHAR2(n)
4,Type numérique,INTEGER / REAL,"NUMBER ou NUMBER(p,s)"
5,Auto-incrément,INTEGER PRIMARY KEY AUTOINCREMENT,SEQUENCE + .NEXTVAL (ou IDENTITY en 12c+)
6,Date/heure serveur,datetime('now'),SYSDATE (via DUAL)
7,Bloc procédural,Non supporté nativement,PL/SQL : DECLARE / BEGIN / END
8,Fonction personnalisée,Non supporté nativement,CREATE OR REPLACE FUNCTION ... RETURN
9,Procédure stockée,Non supporté nativement,CREATE OR REPLACE PROCEDURE


## 13. Connexion Python à Oracle (`python-oracledb`)

Si tu préfères interroger Oracle depuis un notebook Jupyter plutôt que depuis la console DataGrip (utile pour enchaîner avec du pandas, comme dans `03_pandas_advanced.ipynb`), voici le template de connexion. **Cellule non exécutée ici** (nécessite une connexion Oracle active) — à lancer depuis ton propre environnement une fois Oracle XE démarré.

```python
# pip install oracledb

import oracledb
import pandas as pd

connection = oracledb.connect(
    user="system",
    password="ton_mdp",
    dsn="localhost:1521/XEPDB1"
)

# Requête directement vers un DataFrame pandas, comme dans le notebook 03
df = pd.read_sql("SELECT nom, score_je FROM communes ORDER BY score_je DESC", connection)
print(df)

connection.close()
```

> Le mode "thin" de `python-oracledb` (utilisé par défaut depuis la version 1.0) ne nécessite **pas** d'installer le client Oracle complet — un net gain de simplicité par rapport à l'ancien `cx_Oracle`.


## Conclusion

-  `01_sql_basics.ipynb` — terminé
-  `02_sql_advanced.ipynb` — terminé
-  `03_pandas_advanced.ipynb` — terminé
-  `04_oracle_specifics.ipynb` — terminé (ce notebook)
